In [2]:
import csv
import json
import sqlite3

In [3]:
sqlite_db_path = "data/all_hiski_records.sqlite3"
moving_records_with_formats_path = "data/csv/Moving_record_parishes_with_formats_v2.csv"
book_info_json_path = "outputs/json/book_info.json"
missing_books_emigration_path = "outputs/json/missing_books_emigration.json"
missing_books_immigration_path = "outputs/json/missing_books_immigration.json"
emigrate_events_with_books_csv_path = "outputs/csv/emigrate_events_with_books.csv"
immigrate_events_with_books_csv_path = "outputs/json/immigrate_events_with_books.csv"
emigrate_events_with_books_json_path = "outputs/csv/emigrate_events_with_books.json"
immigrate_events_with_books_json_path = "outputs/json/immigrate_events_with_books.json"

In [4]:
connection = sqlite3.connect(sqlite_db_path)
cursor = connection.cursor()

In [5]:
header = []

with open(moving_records_with_formats_path, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    cols = reader.fieldnames[:19]
    

col_to_idx = {cols[i]: i for i in range(len(cols))}
display(col_to_idx)

{'parish_normalized': 0,
 'parish': 1,
 'parish_id': 2,
 'images': 3,
 'doc': 4,
 'url': 5,
 'additional': 6,
 'years': 7,
 'source': 8,
 'archive_id': 9,
 'added': 10,
 'Start_year': 11,
 'End_year': 12,
 'Source_type': 13,
 'added_year': 14,
 'kartta': 15,
 'doctype': 16,
 'Notes': 17,
 'Luokittelu': 18}

In [6]:
cols

['parish_normalized',
 'parish',
 'parish_id',
 'images',
 'doc',
 'url',
 'additional',
 'years',
 'source',
 'archive_id',
 'added',
 'Start_year',
 'End_year',
 'Source_type',
 'added_year',
 'kartta',
 'doctype',
 'Notes',
 'Luokittelu']

In [ ]:
books = []
with open(moving_records_with_formats_path, newline='') as csvfile:
    reader = csv.reader(csvfile)
    
    for row in reader:
        if row[0] == 'parish_normalized' or row[col_to_idx["Start_year"]] == "" or row[col_to_idx["End_year"]] == "":
            continue

        parish_id = int(row[col_to_idx["parish_id"]])
        parish_normalized = row[col_to_idx["parish_normalized"]]
        year_1, year_2 = int(row[col_to_idx["Start_year"]]), int(row[col_to_idx["End_year"]])
        source = row[col_to_idx["source"]]
        archive_id = row[col_to_idx["archive_id"]]
        doc = row[col_to_idx["doc"]]
        luokittelu = row[col_to_idx["Luokittelu"]]

        books.append(
            {
                "parish_id": parish_id, 
                "parish_normalized": parish_normalized, 
                "year_1": year_1, 
                "year_2": year_2, 
                "source": source,
                "archive_id": archive_id,
                "doc": doc,
                "luokittelu": luokittelu
            }
        )

In [10]:
cursor.execute(f"SELECT event_id, parish_id, arr_year, dep_year FROM emigrated")
emigration_events = cursor.fetchall()
emigration_events_dict = {event[0]: {"parish_id": event[1], "arr_year": event[2], "dep_year": event[3], "books": []} for event in emigration_events}

cursor.execute(f"SELECT event_id, parish_id, arr_year, dep_year FROM immigrated")
immigration_events = cursor.fetchall()
immigration_events_dict = {event[0]: {"parish_id": event[1], "arr_year": event[2], "dep_year": event[3], "books": []} for event in immigration_events}

In [11]:
for book in books:
    book["events"] = [
        event_id for event_id, event in emigration_events_dict.items()
        if (
            event["parish_id"] == book["parish_id"]
            and (
                (
                    event["dep_year"] != 0
                    and event["dep_year"] >= book["year_1"]
                    and event["dep_year"] <= book["year_2"]
                )
                or (
                    event["arr_year"] != 0
                    and event["arr_year"] >= book["year_1"]
                    and event["arr_year"] <= book["year_2"]
                )
            )
        )
    ] + [
        event_id for event_id, event in immigration_events_dict.items()
        if (
            event["parish_id"] == book["parish_id"]
            and (
                (
                    event["dep_year"] != 0
                    and event["dep_year"] >= book["year_1"]
                    and event["dep_year"] <= book["year_2"]
                )
                or (
                    event["arr_year"] != 0
                    and event["arr_year"] >= book["year_1"]
                    and event["arr_year"] <= book["year_2"]
                )
            )
        )
    ]

In [12]:
for book in books:
    for event_id in book["events"]:
        if event_id in emigration_events_dict:
            emigration_events_dict[event_id]["books"].append(
                f"{book["parish_normalized"]}/muuttaneet_{book["year_1"]}-{book["year_2"]}_{book["source"]}"
            )
        if event_id in immigration_events_dict:
            immigration_events_dict[event_id]["books"].append(
                f"{book["parish_normalized"]}/muuttaneet_{book["year_1"]}-{book["year_2"]}_{book["source"]}"
            )

In [22]:
with open(book_info_json_path, "w") as fp:
    json.dump(books, fp)

In [8]:
#e_events_has_books = [e for e in emigration_events_dict if len(emigration_events_dict[e]["books"]) > 0]
#e_events_no_books = [e for e in emigration_events_dict if len(emigration_events_dict[e]["books"]) == 0]
e_events_has_books = {e: emigration_events_dict[e] for e in emigration_events_dict if len(emigration_events_dict[e]["books"]) > 0}
e_events_no_books = {e: emigration_events_dict[e] for e in emigration_events_dict if len(emigration_events_dict[e]["books"]) == 0}

i_events_has_books = [e for e in immigration_events_dict if len(immigration_events_dict[e]["books"]) > 0]
i_events_no_books = [e for e in immigration_events_dict if len(immigration_events_dict[e]["books"]) == 0]

print(
    "portion of events that can be associated with books:",
    (len(e_events_has_books) + len(i_events_has_books)) / (len(emigration_events_dict) + len(immigration_events_dict))
)
print(
    "portion of events that can't be associated with books:",
    (len(e_events_no_books) + len(i_events_no_books)) / (len(emigration_events_dict) + len(immigration_events_dict))
)

print(
    "events matched to at least one book:",
    len(e_events_has_books) + len(i_events_has_books)
)
print(
    "events not matched to books:",
    len(e_events_no_books) + len(i_events_no_books)
)

portion of events that can be associated with books: 0.6370560881330704
portion of events that can't be associated with books: 0.3629439118669296
events matched to at least one book: 251662
events not matched to books: 143377


In [9]:
parishes = cursor.execute(
    f"SELECT id, name FROM parish;"
).fetchall()

parish_id_to_parish = {p[0]: p[1] for p in parishes}

In [ ]:
parishes_missing_books = {
    emigration_events_dict[event_id]["parish_id"]
    for event_id in e_events_no_books
}

emigration_parish_missing_books = {
    parish_id_to_parish[parish_id]: {
        "parish_id": parish_id,
        "years": {},
    }
    for parish_id in parishes_missing_books
}

for event_id in e_events_no_books:
    event = emigration_events_dict[event_id]

    parish_text = parish_id_to_parish[event["parish_id"]]

    parish_info = emigration_parish_missing_books[parish_text]
    if max(event["arr_year"], event["dep_year"]) in parish_info["years"].keys():
        parish_info["years"][max(event["arr_year"], event["dep_year"])].append(event_id)
    else:
        parish_info["years"][max(event["arr_year"], event["dep_year"])] = []
        parish_info["years"][max(event["arr_year"], event["dep_year"])].append(event_id)

for event_id, parish_info in emigration_parish_missing_books.items():
    parish_info["years"] = parish_info["years"]

with open(missing_books_emigration_path, "w") as file:
    json.dump(emigration_parish_missing_books, file, ensure_ascii=False)

In [ ]:
parishes_missing_books = {
    immigration_events_dict[event_id]["parish_id"]
    for event_id in i_events_no_books
}

immigration_parish_missing_books = {
    parish_id_to_parish[parish_id]: {
        "parish_id": parish_id,
        "years": {},
        #"events_missing_book": []
    }
    for parish_id in parishes_missing_books
}

for event_id in i_events_no_books:
    event = immigration_events_dict[event_id]

    parish_text = parish_id_to_parish[event["parish_id"]]

    parish_info = immigration_parish_missing_books[parish_text]
    if max(event["arr_year"], event["dep_year"]) in parish_info["years"].keys():
        parish_info["years"][max(event["arr_year"], event["dep_year"])].append(event_id)
    else:
        parish_info["years"][max(event["arr_year"], event["dep_year"])] = []
        parish_info["years"][max(event["arr_year"], event["dep_year"])].append(event_id)

    #parish_info["events_missing_book"].append(event_id)

#for event_id, parish_info in immigration_parish_missing_books.items():
    #parish_info["years"] = list(parish_info["years"])

with open(missing_books_immigration_path, "w") as file:
    json.dump(immigration_parish_missing_books, file, ensure_ascii=False)

In [12]:
[emigration_events_dict[event_id] for event_id in e_events_no_books]

[{'parish_id': 34, 'arr_year': 0, 'dep_year': 1751, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1751, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1751, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1751, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1751, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1751, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1752, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1752, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1752, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1752, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1752, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1752, 'books': []},
 {'parish_id': 34, 'arr_year': 0, 'dep_year': 1752, 'books': []},
 {'parish_id': 369, 'arr_year': 0, 'dep_year': 1737, 'books': []},
 {'parish_id': 369, 'arr_year': 0, 'dep_year': 1737, 'books': []},
 {'paris

In [13]:
books_has_events = [book for book in books if len(book["events"]) > 0]
books_no_events = [book for book in books if len(book["events"]) == 0]

print("portion of books that can be associated with events:", len(books_has_events) / len(books))
print("portion of books that can't be associated with events:", len(books_no_events) / len(books))

portion of books that can be associated with events: 0.10535778496943546
portion of books that can't be associated with events: 0.8946422150305645


In [14]:
i_events = [[e_id] + list(e.values()) for e_id, e in immigration_events_dict.items()]
e_events = [[e_id] + list(e.values()) for e_id, e in emigration_events_dict.items()]

In [ ]:
with open(immigrate_events_with_books_csv_path, "w") as file:
    writer = csv.writer(file)
    writer.writerows(i_events)

with open(emigrate_events_with_books_csv_path, "w") as file:
    writer = csv.writer(file)
    writer.writerows(e_events)

In [ ]:
with open(emigrate_events_with_books_json_path, "w") as json_file:
    json.dump(emigration_events_dict, json_file)

with open(immigrate_events_with_books_json_path, "w") as json_file:
    json.dump(immigration_events_dict, json_file)